# 🌿 Leva-TTS — Evaluation on Colab T4

Reproduce the performance metrics on a **free T4 GPU**:
**RTF**, **TTFA**, **peak VRAM**, **CER/WER** (Whisper round-trip) and **UTMOS** (reference-free MOS).

This clones the repo (the eval script lives there) and runs `scripts/evaluate.py`.

👉 Use a **T4 GPU** runtime. A short sentence set keeps it within Colab limits.

## 1. GPU check

In [ ]:
# Verify a GPU runtime is active (Runtime ▸ Change runtime type ▸ T4 GPU)
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "⚠️ No GPU — switch the runtime to a GPU!")

## 2. Clone the repo + install

Evaluation needs Whisper + UTMOS in addition to the package.

In [ ]:
!git clone -q https://github.com/MohammedAly22/Leva-TTS.git
%cd Leva-TTS
!pip install -q -e .
!pip install -q faster-whisper jiwer            # ASR round-trip + CER/WER
!apt-get -qq install -y espeak-ng ffmpeg > /dev/null
print("✅ ready")

> ⚠️ **If you see a "You must restart the runtime" message after install**, do
> **Runtime ▸ Restart session**, then continue from the next cell (do *not* re-run
> the install cell).

In [ ]:
# (re-enter the repo dir after a restart)
%cd /content/Leva-TTS

## 3. Download the checkpoint

In [ ]:
import leva_tts
leva_tts.download_model("./checkpoints")
print("checkpoint ready")

## 4. Run the evaluation

Runs the full built-in `TEST_SENTENCES` set (Levantine / code-switch / English).
On a T4 this takes a few minutes (Whisper + UTMOS download on first run).

- Add `--optimize` to benchmark the TF32 + `torch.compile` path.
- Add `--no-asr` and/or `--no-utmos` to skip the slow quality metrics and
  measure only latency / RTF / VRAM.

In [ ]:
!python scripts/evaluate.py \
    --checkpoint checkpoints \
    --speaker Mohamed \
    --tag colab_t4

## 5. Inspect the report

In [ ]:
import glob, json
rep = sorted(glob.glob("eval_results/*colab_t4*.json")) or sorted(glob.glob("eval_results/*.json"))
data = json.load(open(rep[-1]))
print(json.dumps(data.get("summary", data), indent=2, ensure_ascii=False)[:2000])

> ⏱️ T4 is slower than the H100 used in the README, so absolute RTF/TTFA will be higher,
> but the **relative** quality metrics (CER/WER/UTMOS) are representative.